# Ricci Finance v10.4 Lecture Notebook

Dynamic 3D Ricci-capital manifold for finance.

Visual encoding: x/y = stable topology, z = Ricci stress, node size = dollar-volume mass, edge width = capital transport, edge color = Ricci curvature, animation = rolling-window evolution.

In [1]:
from helper import *
import pandas as pd


2026-07-08 11:01:50.276288: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/lib/python3.14/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## Build real yfinance market data


In [2]:
tickers = DEFAULT_TICKERS

# Real market data: raw Yahoo Close and Volume.
# This replaces make_demo_market_data(), which was only synthetic lecture data.
prices, volumes, dollar_volume = download_market_data(
    tickers=tickers,
    period='1y',
    interval='1d',
)

returns = prices_to_returns(prices).dropna(axis=1, how='all')
dollar_volume = dollar_volume.reindex(index=returns.index, columns=returns.columns).fillna(0.0)

#print('Last price date:', prices.index[-1])
prices.tail()


Ticker,AAOI,AMAT,AMD,ANET,AVGO,COHR,IONQ,KLAC,LITE,LRCX,MRVL,MU,NBIS,NVDA,PLTR,QBTS,QUBT,RGTI,SMCI,TSM
Date,,,,,,,,,,,,,,,,,,,,
2026-06-30,148.160004,723.000000,580.909973,169.880005,377.750000,394.470001,53.259998,301.709991,858.059998,433.329987,297.890015,1154.290039,276.170013,200.089996,116.669998,23.990000,9.70,19.320000,29.330000,477.570007
2026-07-01,139.000000,650.909973,540.880005,166.619995,369.339996,368.649994,51.400002,266.190002,801.159973,391.260010,272.049988,1032.280029,229.179993,197.580002,125.730003,23.500000,9.43,18.680000,27.650000,444.230011
2026-07-02,120.949997,603.039978,517.820007,159.990005,360.450012,333.359985,49.119999,235.550003,728.320007,351.410004,245.289993,975.559998,215.619995,194.830002,129.300003,22.530001,9.05,17.940001,27.219999,434.160004
2026-07-06,123.360001,592.789978,552.049988,173.279999,373.899994,335.700012,48.869999,233.309998,731.250000,350.200012,249.270004,984.750000,213.020004,195.550003,132.539993,22.559999,9.37,17.959999,27.190001,451.790009
2026-07-07,114.410004,554.500000,516.109985,166.460007,370.779999,314.130005,45.360001,216.470001,698.909973,326.130005,230.699997,938.380005,195.190002,196.929993,134.369995,21.059999,8.69,16.549999,26.250000,432.570007


## Rolling Ricci-capital frames

In [3]:
frames, starts = build_rolling_frames(
    returns=returns, window_size=60, step=5, max_frames=30,
    graph_mode='knn+bridges', k_neighbors=3, min_corr=0.05, max_bridges=3,
    dollar_volume=dollar_volume, use_capital_weighting=True, capital_alpha=0.35,
)
feature_df = rolling_feature_table(frames)
feature_df.tail()


,date,avg_ricci,ricci_std,clusters,largest_component_ratio,nodes,edges,density
25,2026-05-21 00:00:00,0.237626,0.193002,1,1.0,20,47,0.247368
26,2026-06-05 00:00:00,0.224434,0.197987,1,1.0,20,46,0.242105
27,2026-06-12 00:00:00,0.211856,0.207669,1,1.0,20,44,0.231579
28,2026-06-22 00:00:00,0.238175,0.190732,1,1.0,20,46,0.242105
29,2026-07-07 00:00:00,0.198411,0.218702,1,1.0,20,45,0.236842


## 3D dynamic manifold

Run this in Jupyter to get an interactive Plotly animation with Play/Pause and rotation.

In [4]:
base_graph = build_base_graph_for_layout(frames, all_nodes=returns.columns)
positions_3d = compute_stable_layout_3d(base_graph, seed=42, layout_k=0.45)
fig = build_3d_ricci_capital_animation(frames, positions_3d, z_mode='ricci_stress')
fig.show()


## Surgery-risk direction, not actual surgery

In [5]:
fd = frames[-1]
surgery_risk_direction_table(fd.G).head(20)


,u,v,ricciCurvature,distance,edge_capital_flow,surgery_risk_direction,interpretation
38,NVDA,PLTR,-0.299940,1.188522,2.410002e+11,9.342799,possible future separation
35,NBIS,RGTI,-0.299500,1.028988,5.173453e+10,7.602660,possible future separation
21,COHR,LRCX,-0.237966,0.780450,1.178704e+11,4.734538,possible future separation
26,KLAC,TSM,-0.058784,0.753307,1.671452e+11,1.144353,possible future separation
32,LRCX,NBIS,-0.043112,1.044214,9.991577e+10,1.140191,possible future separation
33,LRCX,TSM,-0.022846,0.747160,1.985115e+11,0.444049,possible future separation
34,NBIS,TSM,-0.010184,0.967851,1.481122e+11,0.253521,possible future separation
5,AMAT,MU,0.396623,0.682328,6.457709e+11,0.000000,normal / coherent
0,AAOI,LITE,0.683038,0.632148,1.569024e+11,0.000000,normal / coherent
1,AAOI,COHR,0.441251,0.711201,9.545900e+10,0.000000,normal / coherent
